Here is the exact text transcribed from the "Key Vocabulary" table shown in the image:

---

## Key Vocabulary

| Term | Meaning |
| --- | --- |
| **Chunk** | A small piece of a document (e.g. one paragraph) |
| **Embedding** | A vector (list of numbers) representing text meaning |
| **Vector Database** | A database that stores and searches vectors by similarity |
| **Retrieval** | Finding the most relevant chunks for a given query |
| **Context Injection** | Adding retrieved chunks into the LLM's prompt |
| **Grounding** | Forcing the LLM to answer based on provided facts |
| **Knowledge Base** | The collection of documents the system can retrieve from |

---

*Note: Below the table, the next section header begins with:* **PART 2: Setting Up the Environment**

In [ ]:
!pip install sentence-transformers chromadb groq pandas -q
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

print("All libraries imported successfully.")
print("Ready to build a RAG system.")

In [ ]:
GROQ_API_KEY="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"

os.environ["GROQ_API_KEY"]=GROQ_API_KEY
groq_client=Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized")
print("Note:If you see a authentication error later,double check your API key.")

PART 4: Loading the Knowledge Base

Our knowledge base is the college_notes.csv file from Day 7.

It contains 15 notes across subjects: Data Engineering, Machine Learning, GenAI, and Python.

          Each row in the CSV represents one document (one knowledge chunk).

          What is a Knowledge Base?
          Simple English: A library of documents that the AI can look through when answering questions.

          Analogy: Like a textbook that the AI can open and read before answering your question.

          Technical: A collection of text documents indexed in a vector database for similarity search.

In [ ]:
df=pd.read_csv('college_notes.csv')
print("Shape of dataset:",df.shape)
print("\nColumn names:",df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

In [ ]:
print("Subjects in the dataset:")

print(df['subject'].value_counts())

print("\n Sample of topics:")
print(df[['note_id' , 'subject' , 'topic']].to_string(index=False))

print("\nLength of content (number of characters) for each note:")

In [ ]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())

print("\n Sample of topics:")
print(df[['note_id' , 'subject' , 'topic']].to_string(index=False))

print("\nLength of content (number of characters) for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

#Chunking

In [ ]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]

metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared: {len(documents)}")
print(f"First Document ID : {ids[0]}")
print(f"First metadata: {metadatas[0]}")
print(f"First 100 chars of Document: {documents[0][:100]}...")

In [ ]:
print("Loading embedding model")
print("(Subsequent runs will be faster as the model is cached)")
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')

print("\nEmbedding model loaded successfully")
test_embedding=embedding_model.encode("This is a test sentence.")
print(f"Test embedding shape:{test_embedding.shape}")
print(f"First 5 values of test embedding:{test_embedding[:5]}")

In [ ]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="college_notes_rag")
print("ChromaDB client created")
print(f"Collection name: college_notes_rag")

print(f"Documents in collection so far: {collection.count()}")

In [ ]:
print("Generating embeddings for all 15 notes...")
print("This may take 15-30 seconds...")

embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\nEmbedding matrix shape:{embeddings.shape}")

embeddings_list = embeddings.tolist()
collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas
)

print(f"\n Documents succussfully added to ChromaDB.")
print(f"Total documents in collection: {collection.count()}")

In [ ]:
def retrieve_relevant_chunks(question , top_k=3):
  question_embedding = embedding_model.encode(question).tolist()

  results = collection.query(
      query_embeddings = [question_embedding],
      n_results = top_k
 )

  return results
print("Retrieval Function defined succesfully")
print("Function: retrieve_relevant_chunks(question , top_5=3)")

In [ ]:
test_question="What is ETL and how does it work in data engineering?"
print(f"Test Question: (test_question)")
print("="*60)

results=retrieve_relevant_chunks(test_question,top_k=3)
print("\nTop 3 Retrieved Chunks:")
print("="*60)

for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
    print(f"\nResults {i+1}:")
    print(f" Subject :{meta['subject']}")
    print(f" Topic : {meta['topic']}")
    print(f" Distance : {dist:.4f}")
    print(f" Content : {doc[:120]}...")

Context injection: We take the retrived data and paste them into the LLM's prompt


Here is the exact text transcribed from the "The RAG Prompt Template" section shown in the image:

---

## The RAG Prompt Template

**SYSTEM:**
You are a helpful academic assistant. Answer questions based ONLY on the provided context. If the answer is not in the context, say "I don't have enough information to answer this." Do not use your general training knowledge. Only use the context provided.

**USER:**
Context:
---

Retrieved Document 1

---

Retrieved Document 2

---


Retrieved Document 3


---

Question:

User's Questions

In [ ]:
def build_context_from_results(results):

  context_parts = []

  for i , (doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):

      chunk_text = f"[Source {i+1}:{meta['subject']} - {meta['topic']}]\n{doc}"
      context_parts.append(chunk_text)
  context_str = "\n\n---\n\n".join(context_parts)

  return context_str


In [ ]:
def generate_rag_answer(question, context):

    system_prompt = """
    You are a helpful academic assistant for engineering students.

    You will be given context retrieved from a college knowledge base and a student's question.

    RULES:
    1. Answer ONLY using the information provided in the context below.
    2. If the answer is not found in the context, say exactly:
       "I don't have enough information in my knowledge base to answer this question."
    3. Do not use your general training knowledge.
    4. Keep answers clear, accurate, and beginner-friendly.
    5. Mention which source the information came from when possible.
    """

    user_prompt = f"""
    Context from knowledge base:

    {context}

    ---

    Student's Question: {question}

    Please answer the question based only on the context provided above.
    """

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )

    answer = response.choices[0].message.content
    return answer


print("RAG generation function defined.")

In [ ]:
def ask_college_assistant(question, top_k=3 , verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("=" * 60)
    print("Step 1: Retrieving relevant documents...")

  results = retrieve_relevant_chunks(question, top_k=top_k)

  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base:")
    for i, meta in enumerate(results['metadatas'][0]):
      print(f" {i+1}. {meta['subject']}- {meta['topic']}")
    print("\n Step 2: Building context string...")

  context = build_context_from_results(results)

  if verbose:
    print(f"Context built ({len(context)} characters)")
    print("\n Step 3: Sending to LLM for answer generation...")

  answer = generate_rag_answer(question,context)

  if verbose:
    print("\n" + "=" *60)
    print("ANSWER:")
    print("=" * 60)
    print(answer)
    print("=" * 60)

  return answer

print("Complete RAG pipeline function ready")
print("Function: ask_college_assistant(question , top_k=3 , verbose=True)")

In [ ]:
question_1="What is ETL and what are it's 3 main stages?"

answer_1=ask_college_assistant(question_1,top_k=3,verbose=True)


In [ ]:
question_2="How do embeddings help in building search systems?"
answer_2=ask_college_assistant(question_2,top_k=3,verbose=True)

In [ ]:
question_3="What is the population of Tokyo?"
print("Testing with an out-of-scope question(not in college notes): ")
answer_3=ask_college_assistant(question_3,top_k=3,verbose=True)

Here is the exact text transcribed from the comparison table shown in `image_f34e6d.jpg`:

## Comparison Table

| Feature | Without RAG | With RAG |
| --- | --- | --- |
| Knowledge source | LLM training data (fixed) | Your custom documents (updatable) |
| Hallucination risk | High | Low |
| Can use private data | No | Yes |
| Knowledge cutoff | Yes (training date) | No (you add new docs anytime) |
| Cites sources | No | Yes (you know which chunk was used) |
| Cost | Cheaper (shorter prompts) | Slightly higher (longer prompts with context) |


Here is the exact text transcribed from the notebook interface shown in `image_f33c98.jpg`:

### Real-World Data Engineering RAG Applications

RAG is not just for chatbots. It is widely used in Data Engineering:

---

#### 1. Data Catalog Assistants

Large companies have thousands of datasets. Engineers ask:
*"What does the customer_churn table contain?"*
RAG retrieves from the data catalog documents and answers accurately.

---

#### 2. Pipeline Debugging Assistants

When a data pipeline fails, engineers ask:
*"Why did the ETL job fail with error code 504?"*
RAG retrieves from past incident reports, runbooks, and error logs.

---

#### 3. SQL Generation from Documentation

Engineers ask: *"Write a query to get revenue by region from the sales schema"*
RAG retrieves schema documentation and the LLM generates accurate SQL.

#### 4.Data Governance Q&A

Compilance team ask:"What is our retention policy for PII data?"
RAG retrieves from governance policy documents

In [ ]:
def retrieve_by_subject(question , subject_filter , top_k=2):

  question_embedding = embedding_model.encode(question).tolist()

  results = collection.query(
      query_embeddings = [question_embedding],
      n_results = top_k,
      where = {"subject" : {"$eq" : subject_filter}}
  )

  return results

print("Retrieving only from GenAI subject")
print("=" * 50)

filtered_results = retrieve_by_subject(
    question="How do LLMs generate text?",
    subject_filter="GenAI",
    top_k=2
)
for i,(doc,meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['metadatas'][0]
)):

   print(f"Result {i+1}: [{meta['subject']}] {meta['topic']}")
   print(f" {doc[:100]}...")


### Beginner Questions

**Q1.** What is hallucination in the context of LLMs?

**Q2.** What does RAG stand for? What problem does it solve?

**Q3.** What is the role of a vector database in the RAG pipeline?

---

### Intermediate Questions

**Q4.** What is the difference between the Indexing phase and the Querying phase of RAG?

**Q5.** Why must you use the same embedding model for both documents and queries?

**Q6.** Why is a low temperature (e.g. 0.1) preferred for RAG-based LLM calls?

---

### Coding Questions

**Q7.** Modify the ask_college_assistant() function to also display the distance scores of retrieved chunks in the output

**Q8**  Change the system prompt in generate_rag_answer() to instruct the LLM to always respond in bullet points

**Q9** Add a function that returns only the topic names of retrieved chunks without their full content

Q1. What is hallucination in the context of LLMs?

Answer:
Hallucination occurs when a Large Language Model (LLM) generates information that sounds correct but is actually false, misleading, or not supported by facts.

Example:
If an LLM says a college offers a course that does not exist, it is hallucinating.


Q2. What does RAG stand for? What problem does it solve?

Answer:
RAG stands for Retrieval-Augmented Generation.

It solves the problem of LLMs lacking access to specific or up-to-date information by retrieving relevant documents from an external knowledge base and providing them as context before generating an answer.

Benefits:

Reduces hallucinations
Provides more accurate answers
Uses domain-specific knowledge
Keeps information up-to-date without retraining the model


Q3. What is the role of a vector database in the RAG pipeline?

Answer:
A vector database stores document embeddings (vector representations of text) and enables efficient similarity search.

Role in RAG:

Store embeddings of documents/chunks.
Convert user query into an embedding.
Find the most similar document chunks.
Return relevant chunks to the LLM as context.

Examples:

ChromaDB
Pinecone
Weaviate
FAISS


Intermediate Questions

Q4. What is the difference between the Indexing phase and the Querying phase of RAG?
Indexing Phase	Querying Phase
Happens before users ask questions	Happens when a user asks a question
Documents are collected and processed	User query is processed
Documents are split into chunks	Query is converted to embedding
Embeddings are generated for chunks	Similar chunks are retrieved
Embeddings are stored in vector DB	Retrieved chunks are sent to LLM
Flow

Indexing

Documents
   ↓
Chunking
   ↓
Embeddings
   ↓
Vector Database

Querying

User Query
   ↓
Embedding
   ↓
Similarity Search
   ↓
Relevant Chunks
   ↓
LLM Answer

Q5. Why must you use the same embedding model for both documents and queries?

Answer:
The same embedding model must be used so that documents and queries are represented in the same vector space.

If different models are used:

Vector dimensions may differ.
Similarity calculations become unreliable.
Retrieval quality decreases significantly.

Example:
If documents are embedded using all-MiniLM-L6-v2, queries should also use all-MiniLM-L6-v2.

Q6. Why is a low temperature (e.g., 0.1) preferred for RAG-based LLM calls?

Answer:
A low temperature makes the model more deterministic and focused on the provided context.

Advantages:
Reduces hallucinations
Produces consistent answers
Sticks closely to retrieved documents
Improves factual accuracy
Example:
temperature = 0.1

This is preferred because RAG systems prioritize correctness over creativity.

In [ ]:
def ask_college_assistant():
    question = input("Ask a question: ")

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    print("\nRetrieved Chunks:\n")

    for i, doc in enumerate(results["documents"][0]):
        distance = results["distances"][0][i]

        print(f"Chunk {i+1}")
        print(f"Distance Score: {distance}")
        print(doc)
        print("-" * 50)

    context = "\n".join(results["documents"][0])

    answer = generate_rag_answer(question, context)

    print("\nAnswer:")
    print(answer)

In [ ]:
system_prompt = """
You are a helpful academic assistant for engineering students.

You will be given context retrieved from a college knowledge base and a student's question.

RULES:
1. Answer ONLY using the information provided in the context below.
2. If the answer is not found in the context, say exactly:
   "I don't have enough information in my knowledge base to answer this question."
3. Always respond using bullet points.
4. Do not make up information.
"""

In [ ]:
def get_retrieved_topics(question):
    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    topics = []

    for metadata in results["metadatas"][0]:
        topics.append(metadata["topic"])

    return topics

### Project Description

Build a complete **College Knowledge Assistant** that:

1. Loads the `college_notes.csv` knowledge base
2. Indexes all notes in ChromaDB with embeddings
3. Accepts a student question
4. Retrieves the top 3 relevant notes
5. Injects them as context into a Groq LLM prompt
6. Returns a clear, grounded answer with source citations
7. Handles questions outside the knowledge base gracefully

In [ ]:
import pandas as pd

# This cell successfully loads the 'college_notes.csv' knowledge base,
# completing the first step of the mini-project.
df = pd.read_csv("college_notes.csv")
print(df.head())

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
import chromadb

client = chromadb.Client()
collection = client.create_collection("college_notes.csv")

for i, row in df.iterrows():
    collection.add(
        ids=[str(i)],
        documents=[row["content"]],
        metadatas=[{
            "topic": row["topic"],
            "subject": row["subject"]
        }]
    )

In [ ]:
def retrieve_notes(question):
    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    return results

In [ ]:
from groq import Groq

client = Groq(api_key="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")

def generate_rag_answer(question, context):

    system_prompt = """
    You are a helpful academic assistant.

    Answer ONLY using the provided context.

    If the answer is not present in the context,
    say:
    'I don't have enough information in my knowledge base to answer this question.'
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",
             "content": f"Context:\n{context}\n\nQuestion:\n{question}"}
        ],
        temperature=0.1
    )

    return response.choices[0].message.content

In [ ]:
def get_sources(results):
    return [
        metadata["topic"]
        for metadata in results["metadatas"][0]
    ]

In [ ]:
def ask_college_assistant():

    question = input("Ask a question: ")

    results = retrieve_notes(question)

    context = "\n".join(results["documents"][0])

    answer = generate_rag_answer(question, context)

    sources = get_sources(results)

    print("\nAnswer:")
    print(answer)

    print("\nSources:")
    for source in sources:
        print("-", source)

In [ ]:
ask_college_assistant()